# Billboard Year-End Hot 100 Charts 2000-2018

**Fix note (updated version):**
- Wikipedia now rejects requests that don't send a browser-like `User-Agent`, which caused `pd.read_html(url)` to fail with `HTTP Error 403: Forbidden`. Fixed by fetching the page with `requests` (with headers) and passing the HTML text to `pd.read_html()` instead of the raw URL.
- `DataFrame.append()` was removed in pandas 2.0+. Replaced the repeated `.append()` calls with a list of per-year DataFrames combined once via `pd.concat()` at the end.
- Collapsed the 19 near-identical copy-pasted year cells into a single loop over `years`.
- **New:** this notebook now creates the full `data/` folder structure itself and saves `billboard.csv` directly into it, so the whole pipeline (notebooks 1 → 2 → 3) is reproducible from a clean checkout without any manual file-moving. It also creates the `data/19000-spotify-songs/` folder as a placeholder — you still need to manually download the Kaggle "19,000 Spotify Songs" dataset (`song_info.csv`, `song_data.csv`) and drop the files into that folder, since that dataset isn't produced by any script here.

In [1]:
import pandas as pd
import numpy as np
import requests
import os

## Set Up Project Folder Structure

Creates the `data/` layout that every downstream notebook in this project expects, so nothing needs to be moved by hand afterward:

```
data/
    billboard.csv                     <- produced by this notebook
    19000-spotify-songs/
        song_info.csv                 <- you must download this from Kaggle
        song_data.csv                 <- you must download this from Kaggle
```

In [2]:
DATA_DIR = 'data'
KAGGLE_DIR = os.path.join(DATA_DIR, '19000-spotify-songs')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(KAGGLE_DIR, exist_ok=True)

print(f"Created/verified: {os.path.abspath(DATA_DIR)}")
print(f"Created/verified: {os.path.abspath(KAGGLE_DIR)}")

if not os.path.exists(os.path.join(KAGGLE_DIR, 'song_info.csv')):
    print(
        "\nNOTE: 'song_info.csv' not found in data/19000-spotify-songs/.\n"
        "Download the '19,000 Spotify Songs' dataset from Kaggle and place\n"
        "song_info.csv (and song_data.csv) in that folder before running\n"
        "2_combining_billboard_19000_datasets.ipynb."
    )

Created/verified: c:\Users\ASUS\Desktop\spotify-project\Top100_Analysis_Codes\data
Created/verified: c:\Users\ASUS\Desktop\spotify-project\Top100_Analysis_Codes\data\19000-spotify-songs

NOTE: 'song_info.csv' not found in data/19000-spotify-songs/.
Download the '19,000 Spotify Songs' dataset from Kaggle and place
song_info.csv (and song_data.csv) in that folder before running
2_combining_billboard_19000_datasets.ipynb.


## Setup

A `User-Agent` header is required — Wikipedia blocks requests using the default `urllib`/`requests` user-agent with a `403 Forbidden`.

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

url_template = 'https://en.wikipedia.org/wiki/Billboard_Year-End_Hot_100_singles_of_'

years = [2000, 2001, 2002, 2003, 2004, 2005, 2006, 
         2007, 2008, 2009, 2010, 2011, 2012, 2013,
         2014, 2015, 2016, 2017, 2018]

# For most years the Hot 100 table is the first table on the page (index 0).
# 2012 and 2013's pages have an extra table before it, so the real one is index 1.
table_index_overrides = {2012: 1, 2013: 1}

## Scrape All Years

Replaces the original notebook's 19 copy-pasted cells (one per year, each ending in `billboard100_df = billboard100_df.append(temp_df)`) with a single loop that fetches each page with `requests`, parses it with `pd.read_html(resp.text, ...)`, and collects the cleaned yearly tables in a list. This avoids both the 403 error and the removed `.append()` method.

In [4]:
dfs = []

for year in years:
    url = url_template + str(year)
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()

    table_idx = table_index_overrides.get(year, 0)
    tables = pd.read_html(resp.text, header=0)
    temp_df = tables[table_idx]

    temp_df.columns.values[2] = "Artist"
    temp_df = temp_df.drop(temp_df.columns[0], axis=1)
    temp_df.Artist = [artist.split(' featuring')[0] for artist in temp_df.Artist]
    temp_df.Title = [title.strip('\"') for title in temp_df.Title]
    temp_df['Year'] = year

    dfs.append(temp_df)
    print(f"{year}: scraped {len(temp_df)} rows")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_34380\2446210007.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text, header=0)


2000: scraped 100 rows


C:\Users\ASUS\AppData\Local\Temp\ipykernel_34380\2446210007.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text, header=0)


2001: scraped 100 rows


C:\Users\ASUS\AppData\Local\Temp\ipykernel_34380\2446210007.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text, header=0)


...

C:\Users\ASUS\AppData\Local\Temp\ipykernel_34380\2446210007.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text, header=0)


2016: scraped 100 rows


C:\Users\ASUS\AppData\Local\Temp\ipykernel_34380\2446210007.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text, header=0)


2017: scraped 100 rows
2018: scraped 100 rows


C:\Users\ASUS\AppData\Local\Temp\ipykernel_34380\2446210007.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text, header=0)


## Combine Into a Single DataFrame

Single `pd.concat()` call instead of the removed `DataFrame.append()`.

In [5]:
billboard100_df = pd.concat(dfs, ignore_index=True)

billboard100_df['Year'] = pd.to_numeric(billboard100_df['Year'])
billboard100_df['Title'] = billboard100_df['Title'].astype(str)
billboard100_df['Artist'] = billboard100_df['Artist'].astype(str)

In [6]:
billboard100_df

,Title,Artist,Year
0,Breathe,Faith Hill,2000
1,Smooth,Santana,2000
2,Maria Maria,Santana,2000
3,I Wanna Know,Joe,2000
4,Everything You Want,Vertical Horizon,2000
...,...,...,...
1895,One Number Away,Luke Combs,2018
1896,Powerglide,Rae Sremmurd,2018
1897,IDGAF,Dua Lipa,2018
1898,Mi Gente,J Balvin and Willy William,2018


## Save Output

Saved directly into `data/`, matching the path `2_combining_billboard_19000_datasets.ipynb` expects — no manual copying needed.

In [7]:
output_path = os.path.join(DATA_DIR, 'billboard.csv')
billboard100_df.to_csv(output_path)
print(f"Saved {len(billboard100_df)} rows to {os.path.abspath(output_path)}")

Saved 1900 rows to c:\Users\ASUS\Desktop\spotify-project\Top100_Analysis_Codes\data\billboard.csv


## Optional: Extending to 2019–2023

Kept for reference — commented out in the original. If re-enabled, route these through the same `requests`-based fetch used above (Wikipedia will 403 a raw `pd.read_html(url)` call the same way it did for 2000–2018).

In [8]:
# extra_years = [2019, 2020, 2021, 2022, 2023]
# extra_table_index_overrides = {2023: 1}
#
# extra_dfs = []
# for year in extra_years:
#     url = url_template + str(year)
#     resp = requests.get(url, headers=headers)
#     resp.raise_for_status()
#
#     table_idx = extra_table_index_overrides.get(year, 0)
#     temp_df = pd.read_html(resp.text, header=0)[table_idx]
#
#     temp_df.columns.values[2] = "Artist"
#     temp_df = temp_df.drop(temp_df.columns[0], axis=1)
#     temp_df.Artist = [artist.split(' featuring')[0] for artist in temp_df.Artist]
#     temp_df.Title = [title.strip('\"') for title in temp_df.Title]
#     temp_df['Year'] = year
#
#     extra_dfs.append(temp_df)
#
# billboard100_df = pd.concat([billboard100_df] + extra_dfs, ignore_index=True)
# billboard100_df.to_csv(os.path.join(DATA_DIR, 'billboard.csv'))